# Textbook Quantum Phase Estimation
Adapted from Qualtran v0.7.0. Copyright 2024 Google LLC, Apache-2.0. Frozen commit `bcb17bb25360ed68ce3bc084645db3e929799389`.


In [ ]:
from collections import Counter
from collections.abc import Callable, Mapping
from math import pi
from typing import cast

import numpy as np
import sympy
import torch
from IPython import display as notebook_display

from flagquantum.algorithms.primitives.phase_estimation_window import (
    RectangularWindowState,
)
from flagquantum.algorithms.primitives.textbook_phase_estimation import (
    TextbookPhaseEstimation,
)
from flagquantum.circuit import Circuit
from flagquantum.compiler._component_trace.resource_model import ComponentCost, GateCost
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.core.value_math import SymbolicReal
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot
from flagquantum.qec.resources.estimation import estimate_ftqc_resources

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def show_signature(name: str, signature: Signature) -> None:
    if all(type(register.bitsize) is int for register in signature):
        dot = export_signature_graph_dot(signature, name)
    else:
        # Symbolic declaration labels retain width, encoding and output lifetime.
        rows = {}
        for register in signature:
            rows[register.name + " width"] = str(register.bitsize)
            rows[register.name + " side"] = str(register.side.name)
            rows[register.name + " encoding"] = str(register.dtype.encoding)
            rows.update({key: str(value) for key, value in register.dtype.parameters})
        dot = export_call_graph_dot(name, {}, annotations={name: rows})
    display(SVG(render_call_graph_svg(dot)))


def make_z_qpe(
    theta: SymbolicReal, window: RectangularWindowState
) -> TextbookPhaseEstimation:
    def controlled_power(
        control: int,
        ports: Mapping[str, tuple[int, ...]],
        workspace: tuple[int, ...],
        classical_bits: tuple[int, ...],
        repetitions: int,
        adjoint: bool,
    ) -> tuple[Instruction, ...]:
        if isinstance(theta, sympy.Expr):
            raise ValueError("Resolve the symbolic angle before circuit binding")
        if workspace or classical_bits:
            raise ValueError("The direct controlled-phase emitter uses no work")
        angle = 2 * pi * theta * (-1 if adjoint else 1)
        return (
            Instruction("cphase", (control, ports["q"][0]), {"theta": angle}),
        ) * repetitions

    bound = (
        window.bind()
        if isinstance(window.bitsize, int) and window.bitsize > 0
        else None
    )
    return TextbookPhaseEstimation(
        Signature.build(q=1), window, controlled_power, bound
    )



## `TextbookPhaseEstimation`
Phase Estimation algorithm as presented in Chapter 5.2 of Neilson & Chuang

The bloq implements the following phase estimation circuit, where `ctrl_state_prep` and
`qft_inv` are configurable parameters.

```
       ┌─────────┐                              ┌─────────┐
  |0> -│         │-----------------------@------│         │---M--- [m1]:highest bit
       │         │                       |      │         │
  |0> -│         │-----------------@-----+------│         │---M--- [m2]
       │CtrlState│                 |     |      │ QFT_inv │
  |0> -│  Prep   │-----------@-----+-----+------│         │---M--- [m3]
       │         │           |     |     |      │         │
  |0> -│         │-----@-----+-----+-----+------│         │---M--- [m4]:lowest bit
       └─────────┘     |     |     |     |      └─────────┘
|Psi> -----------------U-----U^2---U^4---U^8---------------------- |Psi>
```

Note that the circuit measures $\varphi$ (in fixed point representation,
so $0 \leq \varphi \leq 1$) s.t. $e^{i\phi}$ is an eigenvalue of $U$ where $\phi = 2\pi\varphi$
is the estimated phase.

The standard textbook version, as described in Ref[1], uses

1. A uniform state preparation via hadamard on all control qubits for `CtrlStatePrep`
2. A textbook QFT inverse algorithm, implemented in `qft`, for `QFT_inv`

Some useful properties of the phase estimation algorithm are given as follows -

## Cost of TextbookPhaseEstimation
The cost of textbook QPE on `m` control qubits is a sum of costs of

1. **CtrlStatePrep** - This typically scales as $\mathcal{O}(m)$. For uniform state preparation,
        the cost is simply $m$ clifford gates.
2. **Controlled-Us** - There are two cases:
    1. If the unitary is fast forwardable; i.e. cost of $U^n$ is independent of $n$, the cost
        of this step is simply $\mathcal{O}(m \text{ cost(C-U)})$
    2. If the unitary is not fast forwardable; the cost of this step is
        $\mathcal{O}((2 ^ {m} - 1) \text{cost(C-U)})$.
4. **QFT_inv** - The textbook version of QFT uses $\mathcal{O}(m^2)$ rotations but this can be
        improved to $\mathcal{O}(m \log{m})$ using approximate QFT constructions.

As seen above, in most cases the dominant cost of phase estimation comes from step 2.B, which
depends exponentially on the number of control bits $m$.

## Choosing number of control bits - $m$.
In the analysis below, we assume the textbook version of phase estimation where `CtrlStatePrep`
is a uniform state preparation. One can obtain smaller values for $m$ when using different
initial states for the control register, like then `LuisPerinaState` implemented in Qualtran.

### Dependence of $m$ using precision $n$ and success probability $\delta$ as the measure of uncertainty
One way to think about the uncertainty in the obtained phase is to consider the problem where
you wish to estimate the phase $\varphi$ upto $n$ bits of precision (i.e. with accuracy
$2^{-n}$) with probability of success $1 - \delta$. In this setup, the expression of $m$ can be
written as (following Eq 5.35 of Ref[1])

$$
    m = n + \left\lceil\log_2\left({2 + \frac{2}{\delta}}\right)\right\rceil
$$

Setting the number of bits $m$ as per the expression above, we get

$$
    Pr\left[|\tilde{\varphi} - \varphi| < \frac{1}{2^n}\right] \geq 1 - \delta
$$

Here $\varphi$ is the true phase and $\tilde{\varphi}$ is the estimated phase.

`TextbookPhaseEstimation(unitary, RectangularWindow.from_precision_and_delta(precision, delta))` method
can be used to instantiate the Bloq with parameters $m$ and $\delta$ as described above.

### Dependence of $m$ using standard deviation $\epsilon$ as the measure of uncertainty
A stronger way to bound the uncertainty in the obtained phase is to bound the variance of the
estimator $\tilde{\varphi}$ by a given parameter $\epsilon$. Following the analysis in Ref[1,2],
we can show that the variance for textbook phase estimation follows the Standard Quantum
Limit(SQL) of

$$
    \sigma[\phi] = 2\pi \sigma[\tilde{\varphi}] = 2\pi\sqrt{\text{Var}[\tilde{\varphi}]}
                                                \leq\frac{\pi}{\sqrt{M}}=\frac{\pi}{\sqrt{2^m}}
$$

Therefore, to bound the standard deviation of the phase estimator $\tilde{\phi}$ by given parameter
$\epsilon$, we set

$$
    m = \left\lceil2\log_2 \left(\frac{\pi}{\epsilon}\right)\right\rceil
$$

`TextbookPhaseEstimation(unitary, RectangularWindow.from_standard_deviation_eps(eps))` method can be
used to instantiate the Bloq with parameter $\epsilon$ as described above.


#### Parameters
 - `unitary`: Bloq representing the unitary to run the phase estimation protocol on.
 - `ctrl_state_prep`: Bloq to prepare the control state on the phase register.
 - `qft_inv`: Bloq to apply inverse QFT on the phase register. Defaults to `qft(self.m_bits).adjoint()`  

#### Registers
 - `qpe_reg`: Control register of type `QFxp(self.m_bits, self.m_bits)` for phase estimation.
 - `target registers`: All registers used in `self.unitary.signature`  

#### References
 - [Quantum Computation and Quantum Information: 10th Anniversary Edition, Nielsen & Chuang](https://www.cambridge.org/highereducation/books/quantum-computation-and-quantum-information/01E10196D0A682A6AEFFEA52D53BE9AE#overview). Chapter 5.2
 - [Entanglement-free Heisenberg-limited phase estimation](https://arxiv.org/abs/0709.2996). 


In [ ]:
# TextbookPhaseEstimation composes the accepted named child interfaces.


### Example Instances

In [ ]:
textbook_qpe_small = make_z_qpe(0.234, RectangularWindowState(3))


In [ ]:
theta = sympy.Symbol("theta")
m_bits = sympy.Symbol("m")
textbook_qpe_using_m_bits = make_z_qpe(theta, RectangularWindowState(m_bits))


In [ ]:
theta = sympy.Symbol("theta")
epsilon = sympy.symbols("epsilon")
textbook_qpe_from_standard_deviation_eps = make_z_qpe(
    theta, RectangularWindowState.from_standard_deviation_eps(epsilon)
)


In [ ]:
theta = sympy.Symbol("theta")
precision, delta = sympy.symbols("n, delta")
textbook_qpe_from_precision_and_delta = make_z_qpe(
    theta, RectangularWindowState.from_precision_and_delta(precision, delta)
)


#### Graphical Signature

In [ ]:
for name, operation in (
    ("textbook_qpe_small", textbook_qpe_small),
    ("textbook_qpe_using_m_bits", textbook_qpe_using_m_bits),
    (
        "textbook_qpe_from_standard_deviation_eps",
        textbook_qpe_from_standard_deviation_eps,
    ),
    ("textbook_qpe_from_precision_and_delta", textbook_qpe_from_precision_and_delta),
):
    show_signature(name, operation.signature)


### Call Graph

In [ ]:
textbook_qpe_graph = textbook_qpe_small.source_graph(
    root="TextbookPhaseEstimation",
    preparation=("RectangularWindow", 3),
    controlled_unitary=("CZPow", 0.468),
    inverse_qft=("TextbookQFT†", 3),
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                textbook_qpe_graph.root, textbook_qpe_graph.direct_calls
            )
        )
    )
)
display(textbook_qpe_graph.leaf_counts)


## Appendix

### Derivation of variance for textbook phase estimation 

When using a phase register of size  $m$ and uniform superposition state as the initial state, textbook phase estimation ends with a state

$$
    \frac{1}{2^m}\sum_{k, l=0}^{2^m-1}e^\frac{-2\pi i k l}{2^m} e^{2 \pi i \varphi k} |l\rangle
$$

Let $b$ be an integer in the range $[0, 2^m)$ that represents the best $m$-bit estimate of the phase $\varphi$ s.t. $\varphi = b / 2^m + \delta$ where $0 \leq \delta \leq 2^{-m}$. 

Let $\alpha_l$ be the amplitude of state $|(b + l) \text{ mod } 2^m\rangle$, then $\alpha_{l}$ can be written as 

$$
    \alpha_l = \frac{1}{2^m} \left( \frac{1 - e^{2\pi i \left(2^m \delta - l\right)}}
                                         {1 - e^{2\pi i \left(\delta - l/2^m \right)}} \right)
$$

For any real $\theta$, $|1 - e^{i \theta}| \leq 2$, so

$$
    |\alpha_l| \leq \frac{2}{2^m | 1 - e^{2\pi i \left(\delta - l/2^m \right)} | }
$$

Whenever $-\pi \leq \theta \leq \pi$, we have $|1 - e^{i \theta}| \geq \frac{2 |\theta|}{\pi}$ and when $−2^{m−1} \lt l \leq 2^{m−1}$ we have $-\pi \leq 2\pi \left(\delta - l/2^m \right) \leq \pi$. Therefore, we have

$$
    |\alpha_l| \leq \frac{1}{2^{m + 1} |\left(\delta - l/2^m \right)| }
$$

Let $\tilde{\varphi}$ be a random variable that represents the estimated phase. We have

$$
\begin{align}
Var[\tilde{\varphi}]&\leq \sum_{l=-2^{m-1} + 1}^{2^{m-1}} |\alpha_l|^2 \left(\frac{b + l}{2^m} - \varphi\right)^2 \\
                    &= \sum_{l=-2^{m-1} + 1}^{2^{m-1}} |\alpha_l|^2 \left(\delta - l/2^m \right)^2 \\ 
                    &= \sum_{l=-2^{m-1} + 1}^{2^{m-1}} \frac{1}{2^{2(m + 1)} |\left(\delta - l/2^m \right)|}\left(\delta - l/2^m \right)^2 \\
                    &= \sum_{l=-2^{m-1} + 1}^{2^{m-1}} \frac{1}{2^{2(m + 1)}} \\
                    &= \frac{2^m}{2^{2(m + 1)}} = \frac{1}{4 2^{m}} = \frac{1}{4 M}\\
\end{align}
$$

Therefore, we get

$$
    \sigma[\tilde{\phi}] = 2\pi\sigma[\tilde{\varphi}] = \frac{\pi}{\sqrt{M}}
$$

## `RectangularWindowState`
Window state used in Textbook version of QPE. Applies Hadamard on all qubits.

#### Parameters
 - `bitsize`: Size of the control register to prepare window state on. 

#### Registers
 - `qpe_reg`: A `bitsize` sized RIGHT register.


In [ ]:
# RectangularWindowState reuses the shared H preparation.


### Example Instances

In [ ]:
rectangular_window_state_small = RectangularWindowState(5)


In [ ]:
rectangular_window_state_symbolic = RectangularWindowState(sympy.Symbol("n"))


#### Graphical Signature

In [ ]:
show_signature(
    "rectangular_window_state_small", rectangular_window_state_small.signature
)
show_signature(
    "rectangular_window_state_symbolic", rectangular_window_state_symbolic.signature
)


### Call Graph

In [ ]:
rectangular_window_graph = rectangular_window_state_small.source_graph()
rectangular_direct = {
    edge: count
    for edge, count in rectangular_window_graph.direct_calls.items()
    if edge[0] == rectangular_window_graph.root
}
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(rectangular_window_graph.root, rectangular_direct)
        )
    )
)
display({child: count for (_, child), count in rectangular_direct.items()})


In [ ]:
rectangular_window_state_symbolic = RectangularWindowState(sympy.Symbol("n"))


## Append and separate accounts

Symbolic signatures above are declaration diagrams. Physical execution below resolves three phase bits and theta=0.234. The direct callback repeats each controlled phase according to the frozen Power schedule. Its primitive inventory and the independently declared measured CZPow model are displayed separately. The original appendix is retained, including its original variance derivation conventions. SDK-default controlled-Bloq synthesis and native walk/GQSP admission gaps remain in the source-file audit.


In [ ]:
circuit = Circuit(4, dtype=torch.complex128)
circuit.x(3)
circuit.append(textbook_qpe_small.make_on(phase=(0, 1, 2), target=(3,)))
phase_amplitudes = circuit.state().detach().numpy().reshape(8, 2)[:, 1]
indices = np.arange(8)
expected = (
    np.exp(-2j * np.pi * np.outer(indices, indices) / 8)
    @ np.exp(2j * np.pi * 0.234 * indices)
    / 8
)
np.testing.assert_allclose(phase_amplitudes, expected, atol=1e-13, rtol=0)
assert np.argmax(np.abs(phase_amplitudes) ** 2) == 2
primitive_counts = dict(Counter(g.name for g in circuit.to_ir().instructions[1:]))
model = textbook_qpe_small.resource_model(
    ComponentCost("RectangularWindow", 3, GateCost(clifford=3)),
    ComponentCost(
        "CZPow", 3, GateCost(and_bloq=1, clifford=1, rotation=1, measurement=1)
    ),
)
declared_model = estimate_ftqc_resources(model)
display(primitive_counts)
display(declared_model.logical)
